In [16]:
!pip install transformers datasets pykakasi korean-romanizer evaluate seqeval

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 67.8 MB/s  0:00:00
  DEPRECATION: Building 'seqeval' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'seqeval'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16251 sha256=eda22a9e0bae5dc1ff820f8c26d7f16801f62779da675242cb2659d50832625f
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [seqeval]m3/5 [scikit-learn]

[notice] A new release of pip is available: 25.2 -> 26.1.2
[not

In [29]:
import os
import numpy as np
import pykakasi
from korean_romanizer.romanizer import Romanizer
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
import evaluate

In [30]:
%time

CPU times: user 21 μs, sys: 1e+03 ns, total: 22 μs
Wall time: 45.1 μs


In [31]:
#Initializing transliteration tools
def transliterate_token(token: str, lang: str) -> str:
    if not token or token.isspace():
        return token
    if lang == 'ja':
        kks = pykakasi.kakasi()
        converted = kks.convert(token)
        return "".join([item['hepburn'] for item in converted]) if converted else token
    elif lang == 'ko':
        try:
            return Romanizer(token).romanize()
        except Exception:
            return token
    return token

In [32]:
# Loading WikiANN datasets
languages = ['en', 'de', 'fi', 'ja', 'ko']
train_splits = []
val_splits = []

for lang in languages:
    print(f"Processing language: {lang}")
    ds = load_dataset("wikiann", lang)
    
    # Apply transliteration to Japanese and Korean
    if lang in ['ja', 'ko']:
        def process_row(example):
            example['tokens'] = [transliterate_token(t, lang) for t in example['tokens']]
            return example
        ds = ds.map(process_row)
        
    train_splits.append(ds['train'])
    val_splits.append(ds['validation'])

Processing language: en
Processing language: de
Processing language: fi
Processing language: ja
Processing language: ko


In [33]:
train_dataset = concatenate_datasets(train_splits).shuffle(seed=42)
val_dataset = concatenate_datasets(val_splits).shuffle(seed=42)

In [34]:
#Loading tokenizer and aligning labels
model_checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)

label_names = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]

In [35]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True, max_length=128
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

train_tokenized = train_dataset.map(tokenize_and_align_labels, batched=True)
val_tokenized = val_dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [36]:
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_names[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [37]:
#Initializing model and Trainer
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint, 
    num_labels=len(label_names)
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# With 8 hours, we can comfortably run 4 epochs with a standard learning rate
training_args = TrainingArguments(
    output_dir="./roberta-pii-model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    fp16=True, # Critical for GPU speed
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none" # Disables weights & biases logging
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [42]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], 
        truncation=True, 
        is_split_into_words=True, 
        max_length=512 # <--- INCREASED TO MAX
    )
    # ... (keep the rest of the alignment logic the same) ...

# 2. Update Training Arguments (Max out the V100)
training_args = TrainingArguments(
    output_dir="./shared/roberta-pii-model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,               # Slightly higher LR for bigger batches
    per_device_train_batch_size=128,   # <--- CRANK THIS UP (Try 64 or 128)
    per_device_eval_batch_size=64,    # <--- CRANK THIS UP
    dataloader_num_workers=4,         # <--- STOPS GPU STARVATION
    dataloader_pin_memory=True,       # <--- FASTER TRANSFER
    num_train_epochs=15,              # <--- TRAIN FURTHER
    weight_decay=0.01,
    fp16=True,                        
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

# 3. Add Early Stopping to the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # <--- PREVENTS OVERFITTING
)

In [43]:
print("Starting training!")
trainer.train()

print("Training complete. Saving model...")
trainer.save_model("./shared/final_roberta_pii")
print("Model saved to ./final_roberta_pii")

Starting training!


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.491900,0.315160,0.715000,0.712613,0.713805,0.892850
2,0.296500,0.278492,0.740667,0.757977,0.749222,0.904795
3,0.259200,0.275992,0.757873,0.775822,0.766742,0.905657
4,0.211200,0.262465,0.781702,0.788566,0.785119,0.914929
5,0.185300,0.260563,0.782968,0.796394,0.789624,0.916644
6,0.151800,0.276261,0.787305,0.799520,0.793365,0.915261
7,0.135400,0.297643,0.786758,0.805615,0.796075,0.916369
8,0.108100,0.313611,0.780448,0.808257,0.794109,0.915066
9,0.093300,0.327404,0.785358,0.811255,0.798096,0.915844
10,0.081100,0.341899,0.787999,0.818686,0.803049,0.917805


Training complete. Saving model...
Model saved to ./final_roberta_pii


In [44]:
import shutil

# Replace 'my_zipped_file' with the name you want for your zip archive
# Replace 'folder_to_zip' with the path to the folder you want to compress
shutil.make_archive('final_roberta_pii', 'zip', './shared/final_roberta_pii')

'/workspace/final_roberta_pii.zip'

In [1]:
http://0.0.0.0:8000/docs

SyntaxError: invalid syntax (3977238879.py, line 1)